# 13 — Last.fm Popularity Feature

Builds `album_lastfm_popularity_matrix.npz` from the scraped Last.fm parquet.

**Input:** `data/lastfm_data.parquet` (produced by merging all worker parquets in `1-data/05-lastfm-scraper.ipynb`)

**Output:** `data/features/album_lastfm_popularity_matrix.npz`
- 2 columns: `album_scrobbles`, `artist_scrobbles` — both **log1p then min-max** scaled to [0, 1]
- Same row order as `data/features/album_ids.pkl` (1,758,488 albums)
- Albums not in Last.fm data → all zeros (sparse)

**Design notes:**
- **Why only 2 of the 4 scraped metrics?** Listeners and scrobbles correlate at r ≈ 0.9+ (see
  `2-eda/03-EDA-popularity.ipynb`). Carrying both inflated the block's share of the weighted
  cosine without adding information, so the listener columns are dropped. `album_scrobbles` is
  the per-album signal; `artist_scrobbles` acts as a broader popularity proxy.
- **Why log1p before min-max?** Raw scrobbles are extremely long-tailed — plain min-max maps the
  top of the tail (Beatles, Radiohead) to 1.0 and compresses nearly everything else into
  [0, 0.1]. Log-scaling spreads mid-popularity albums across the usable range, which also makes
  the app's Popularity knob feel responsive instead of all-or-nothing.
- **Matching is two-pass:** exact join on normalised (artist, album) names, then a fuzzy fallback
  (rapidfuzz `token_sort_ratio ≥ 90`, artist must match exactly) for rows that miss on
  punctuation, articles, or edition suffixes.
- **Coverage** (last full run): 252,167 albums with signal — 14.3% of the master index. Exact
  matching does almost all the work; the fuzzy pass adds a couple hundred albums. The gap is
  mostly albums that were never scraped, not name mismatches.
- **Requires:** `rapidfuzz` (in `requirements.txt`).

In [1]:
import os, re, pickle
import pandas as pd
import numpy as np
from scipy.sparse import lil_matrix, save_npz, load_npz

DATA_DIR     = '../data'
FEATURES_DIR = '../data/features'
INPUT_PATH   = os.path.join(DATA_DIR, 'lastfm_data.parquet')

## 1. Load the scraped parquet

In [2]:
df = pd.read_parquet(INPUT_PATH)
print(f'Rows loaded: {len(df):,}')
print(df.dtypes)
df.head(3)

Rows loaded: 207,893
Artist              object
Album               object
Artist_Listeners    object
Artist_Scrobbles    object
Album_Listeners     object
Album_Scrobbles     object
Similar_Artists     object
Artist_URL          object
Album_URL           object
dtype: object


,Artist,Album,Artist_Listeners,Artist_Scrobbles,Album_Listeners,Album_Scrobbles,Similar_Artists,Artist_URL,Album_URL
0,!!!,!!!,"721,888","13,594,661","92,549","618,283","The Rapture, The Juan Maclean, Fujiya & Miyagi...",https://www.last.fm/music/!!!,https://www.last.fm/music/!!!/!!!
1,!!!,Louden Up Now,"721,527","13,589,664","195,585","2,063,840","The Rapture, Fujiya & Miyagi, The Juan Maclean...",https://www.last.fm/music/!!!,https://www.last.fm/music/!!!/Louden+Up+Now
2,!Action Pact!,Mercury Theatre - On the Air / Survival of the...,"5,447","78,895",None,None,"The Expelled, Violators, Hagar the Womb, Lost ...",https://www.last.fm/music/!Action+Pact!,https://www.last.fm/music/!Action+Pact!/Mercur...


In [3]:
# Rename to safe internal names
df = df.rename(columns={
    'Artist':           'artist_name',
    'Album':            'album_name',
    'Artist_Listeners': 'artist_listeners',
    'Artist_Scrobbles': 'artist_scrobbles',
    'Album_Listeners':  'album_listeners',
    'Album_Scrobbles':  'album_scrobbles',
    'Similar_Artists':  'similar_artists',
})

df = df[['artist_name', 'album_name',
         'album_listeners', 'album_scrobbles',
         'artist_listeners', 'artist_scrobbles']].copy()

# Convert numeric cols — handles commas ("1,234,567") and all null-like strings
for col in ['album_listeners', 'album_scrobbles', 'artist_listeners', 'artist_scrobbles']:
    df[col] = (
        df[col].astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()
        .replace({'': np.nan, 'N/A': np.nan, 'None Found': np.nan, 'None': np.nan})
        .astype(float)
        .fillna(0)
    )

print(f'After cleaning: {len(df):,} rows')
df.describe()

After cleaning: 207,893 rows


,album_listeners,album_scrobbles,artist_listeners,artist_scrobbles
count,2.078930e+05,2.078930e+05,2.078930e+05,2.078930e+05
mean,1.684856e+04,2.239418e+05,4.053909e+05,1.468322e+07
std,9.673848e+04,2.209101e+06,9.533021e+05,6.695538e+07
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.900000e+01,4.860000e+02,5.160000e+03,5.581600e+04
50%,5.470000e+02,5.766000e+03,4.304300e+04,5.058660e+05
75%,4.438000e+03,4.065000e+04,2.625230e+05,3.862833e+06
max,5.109393e+06,2.583912e+08,9.168262e+06,1.566143e+09


## 2. Normalise names & deduplicate

In [4]:
def normalise(s):
    if not isinstance(s, str): return ''
    s = s.lower()
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

df['artist_norm'] = df['artist_name'].map(normalise)
df['album_norm']  = df['album_name'].map(normalise)

# Keep highest-scrobbles row per (artist, album)
df = (
    df.sort_values('album_scrobbles', ascending=False)
      .drop_duplicates(subset=['artist_norm', 'album_norm'])
      .reset_index(drop=True)
)
print(f'After dedup: {len(df):,} unique (artist, album) pairs')

After dedup: 205,850 unique (artist, album) pairs


## 3. Match to MusicBrainz album_ids

In [5]:
lookup = (
    pd.read_parquet(os.path.join(DATA_DIR, 'mb_album_artists.parquet'),
                    columns=['album_id', 'album_name', 'artist_name'])
    .drop_duplicates(subset='album_id')
    .reset_index(drop=True)
)
lookup['artist_norm'] = lookup['artist_name'].map(normalise)
lookup['album_norm']  = lookup['album_name'].map(normalise)

merged = df.merge(
    lookup[['album_id', 'artist_norm', 'album_norm']],
    on=['artist_norm', 'album_norm'],
    how='inner'
)

print(f'Last.fm rows         : {len(df):,}')
print(f'Matched to album_id  : {len(merged):,}  ({100*len(merged)/len(df):.1f}%)')
merged.head(3)

Last.fm rows         : 205,850
Matched to album_id  : 252,047  (122.4%)


,artist_name,album_name,album_listeners,album_scrobbles,artist_listeners,artist_scrobbles,artist_norm,album_norm,album_id
0,Radiohead,OK Computer,4802669.0,258391199.0,8297618.0,1.392329e+09,radiohead,ok computer,37119
1,Mariah Carey,#1’s,5109393.0,256842842.0,5109393.0,2.568428e+08,mariah carey,1 s,36340
2,Radiohead,The Bends,3664435.0,178367971.0,8297618.0,1.392329e+09,radiohead,the bends,215261


In [6]:
# Sample unmatched rows — useful to spot naming issues
matched_keys = set(zip(merged['artist_norm'], merged['album_norm']))
unmatched = df[
    ~df.apply(lambda r: (r['artist_norm'], r['album_norm']) in matched_keys, axis=1)
].head(10)
print('Sample unmatched:')
print(unmatched[['artist_name', 'album_name']].to_string())

Sample unmatched:
         artist_name                     album_name
294      The Beatles           Magical Mystery Tour
313           Prince                    Purple Rain
315      Linkin Park                    Reanimation
540           Fugazi          Instrument Soundtrack
839    Elvis Presley                    Blue Hawaii
853       Pink Floyd             Obscured by Clouds
906              Air            The Virgin Suicides
915   Arctic Monkeys          Beneath the Boardwalk
950      Evanescence                         Origin
1003        Therapy?  Live @ Sudoeste Festival 1998


## 3b. Fuzzy-match fallback for unmatched rows

Exact matching misses rows where the names differ by punctuation, articles, or edition
suffixes. For each unmatched Last.fm row, candidates must share the exact `artist_norm`;
the album name is scored with rapidfuzz `token_sort_ratio` and accepted at ≥ 90. Keeping the
artist exact (not fuzzy) is what keeps false positives near zero — the threshold only has to
disambiguate among one artist's own discography.

In [7]:
from rapidfuzz import process, fuzz

unmatched_all = df[
    ~df.apply(lambda r: (r['artist_norm'], r['album_norm']) in matched_keys, axis=1)
].copy()

# Group lookup candidates by artist_norm for fast retrieval
lookup_by_artist = (
    lookup.groupby('artist_norm')[['album_id', 'album_norm']]
    .apply(lambda x: x.to_dict('records'))
    .to_dict()
)

fuzzy_rows = []
for _, row in unmatched_all.iterrows():
    candidates = lookup_by_artist.get(row['artist_norm'], [])
    if not candidates:
        continue
    album_choices = [c['album_norm'] for c in candidates]
    result = process.extractOne(row['album_norm'], album_choices, scorer=fuzz.token_sort_ratio)
    if result is None:
        continue
    match_str, score, idx = result
    if score >= 90:
        fuzzy_rows.append({**row.to_dict(), 'album_id': candidates[idx]['album_id']})

fuzzy_df = pd.DataFrame(fuzzy_rows) if fuzzy_rows else pd.DataFrame(columns=merged.columns)
print(f'Fuzzy matches recovered : {len(fuzzy_df):,}')

merged = pd.concat([merged, fuzzy_df], ignore_index=True).drop_duplicates(subset='album_id')
print(f'Total after fuzzy       : {len(merged):,}  ({100*len(merged)/len(df):.1f}% of scraped)')

Fuzzy matches recovered : 236
Total after fuzzy       : 252,167  (122.5% of scraped)


## 4. Log-transform + min-max scale (2 columns)

Only the two scrobbles columns are kept — listeners and scrobbles correlate at r ≈ 0.9+, so
the listener columns add cosine weight without information. `log1p` is applied before min-max
because raw scrobbles are long-tailed: without it, the most-scrobbled albums pin the scale and
nearly the entire catalogue lands in [0, 0.1]. After log-scaling, mid-popularity albums spread
across the full [0, 1] range.

In [8]:
FEATURE_COLS = ['album_scrobbles', 'artist_scrobbles']

def log_minmax(s):
    s_log = np.log1p(s)
    mn, mx = s_log.min(), s_log.max()
    return s_log * 0.0 if mx == mn else (s_log - mn) / (mx - mn)

for col in FEATURE_COLS:
    merged[col + '_scaled'] = log_minmax(merged[col])

merged[[c + '_scaled' for c in FEATURE_COLS]].describe().round(3)

,album_scrobbles_scaled,artist_scrobbles_scaled
count,252167.000,252167.000
mean,0.501,0.645
std,0.208,0.154
min,0.000,0.000
25%,0.379,0.548
50%,0.508,0.665
75%,0.666,0.793
max,1.000,1.000


## 5. Build sparse matrix

Rows follow `album_ids.pkl` exactly — the shared row order across every feature block, required
for the horizontal stack and the app's `weighted_cosine`. Unmatched albums keep all-zero rows.

In [9]:
with open(os.path.join(FEATURES_DIR, 'album_ids.pkl'), 'rb') as f:
    album_ids = np.asarray(pickle.load(f))
album_id_to_row = {int(a): i for i, a in enumerate(album_ids)}

mat = lil_matrix((len(album_ids), 2), dtype=np.float32)
hits = 0
for _, row in merged.iterrows():
    idx = album_id_to_row.get(int(row['album_id']))
    if idx is None: continue
    mat[idx, 0] = row['album_scrobbles_scaled']
    mat[idx, 1] = row['artist_scrobbles_scaled']
    hits += 1

mat = mat.tocsr()
print(f'Albums with Last.fm data : {hits:,} / {len(album_ids):,}  ({100*hits/len(album_ids):.3f}%)')
print(f'Matrix shape             : {mat.shape},  nnz={mat.nnz:,}')

Albums with Last.fm data : 252,167 / 1,758,488  (14.340%)
Matrix shape             : (1758488, 2),  nnz=493,056


## 6. Save

In [10]:
NPZ_PATH = os.path.join(FEATURES_DIR, 'album_lastfm_popularity_matrix.npz')
save_npz(NPZ_PATH, mat)
print(f'NPZ saved  : {NPZ_PATH}')

# Save matched table as parquet for reference
MATCHED_PATH = os.path.join(DATA_DIR, 'lastfm_album_matched.parquet')
merged.to_parquet(MATCHED_PATH, index=False)
print(f'Matched    : {MATCHED_PATH}')

NPZ saved  : ../data/features/album_lastfm_popularity_matrix.npz


Matched    : ../data/lastfm_album_matched.parquet


## 7. Sanity check

In [11]:
test   = load_npz(NPZ_PATH)
scores = np.asarray(test[:, 0].todense()).ravel()   # col 0 = album_scrobbles
top10  = np.argsort(-scores)[:10]
lk     = lookup.set_index('album_id')

print(f'Shape : {test.shape}')
print(f'NNZ   : {test.nnz:,}')
print('\nTop 10 by album scrobbles:')
for i, idx in enumerate(top10):
    aid = int(album_ids[idx])
    r   = lk.loc[aid] if aid in lk.index else {'album_name': '?', 'artist_name': '?'}
    print(f'  {i+1:2d}. {r["artist_name"]} -- {r["album_name"]}  ({scores[idx]:.4f})')

Shape : (1758488, 2)
NNZ   : 493,056

Top 10 by album scrobbles:
   1. Radiohead -- OK Computer  (1.0000)
   2. Mariah Carey -- #1’s  (0.9997)
   3. Radiohead -- The Bends  (0.9809)
   4. System of a Down -- Toxicity  (0.9771)
   5. My Chemical Romance -- Three Cheers for Sweet Revenge  (0.9751)
   6. Kanye West -- Late Registration  (0.9712)
   7. Coldplay -- Parachutes  (0.9661)
   8. Kanye West -- The College Dropout  (0.9608)
   9. Linkin Park -- Meteora  (0.9598)
  10. Radiohead -- Kid A  (0.9598)


## 8. Coverage summary

Run-to-run regression check: if the scraper or the matching steps change, these numbers move.
Exact matching should stay ≈ 99% of total matches; a big jump in the fuzzy share means the
exact join broke (e.g. a normalisation change on one side only).

In [12]:
n_total       = len(album_ids)
n_fuzzy       = len(fuzzy_df)
n_exact       = len(merged) - n_fuzzy
n_in_matrix   = hits
n_missing     = n_total - n_in_matrix

print("--- Last.fm popularity coverage ---")
print(f"Scraped Last.fm rows     : {len(df):,}")
print(f"  Exact name matches     : {n_exact:,}")
print(f"  Fuzzy matches added    : {n_fuzzy:,}")
print(f"  Total matched          : {len(merged):,}")
print()
print(f"Albums in master index   : {n_total:,}")
print(f"  With Last.fm signal    : {n_in_matrix:,}  ({100*n_in_matrix/n_total:.2f}%)")
print(f"  No signal (all zeros)  : {n_missing:,}  ({100*n_missing/n_total:.2f}%)")

--- Last.fm popularity coverage ---
Scraped Last.fm rows     : 205,850
  Exact name matches     : 251,931
  Fuzzy matches added    : 236
  Total matched          : 252,167

Albums in master index   : 1,758,488
  With Last.fm signal    : 252,167  (14.34%)
  No signal (all zeros)  : 1,506,321  (85.66%)
